# Mejoras a la Fusión Propuesta — Estudio de Ablación
Comparamos 3 estrategias de mejora sobre la arquitectura base de fusión intermedia:

1. **Base** (referencia) — concatenación + BCE
2. **Focal Loss** — para el desbalance de clases críticas (MEL)
3. **Tuning de hiperparámetros** — búsqueda sobre proj_dim, dropout, lr
4. **Atención entre modalidades** — fusión con pesos aprendidos

**Regla metodológica:** la selección de la mejor configuración se hace SIEMPRE
por el AUC de validación. El test se evalúa una sola vez al final.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
torch.manual_seed(42)
np.random.seed(42)

Dispositivo: cuda


## 1. Cargar datos (igual que la propuesta base)

In [ ]:
BASE_PATH    = '/content/drive/MyDrive/PFC1 - Noemi/dataset/PAD-UFES-20'
RESULTS_PATH = os.path.join(BASE_PATH, 'resultados')

clip_train = np.load(os.path.join(RESULTS_PATH, 'clip_X_train.npy'))
clip_val   = np.load(os.path.join(RESULTS_PATH, 'clip_X_val.npy'))
clip_test  = np.load(os.path.join(RESULTS_PATH, 'clip_X_test.npy'))
dino_train = np.load(os.path.join(RESULTS_PATH, 'dino_X_train.npy'))
dino_val   = np.load(os.path.join(RESULTS_PATH, 'dino_X_val.npy'))
dino_test  = np.load(os.path.join(RESULTS_PATH, 'dino_X_test.npy'))
y_train = np.load(os.path.join(RESULTS_PATH, 'y_train.npy'))
y_val   = np.load(os.path.join(RESULTS_PATH, 'y_val.npy'))
y_test  = np.load(os.path.join(RESULTS_PATH, 'y_test.npy'))

df_train = pd.read_csv(os.path.join(BASE_PATH, 'split_train.csv'))
df_val   = pd.read_csv(os.path.join(BASE_PATH, 'split_val.csv'))
df_test  = pd.read_csv(os.path.join(BASE_PATH, 'split_test.csv'))

META_COLS = ['smoke','drink','pesticide','gender','skin_cancer_history','cancer_history',
             'has_piped_water','has_sewage_system','age','fitspatrick','diameter_1','diameter_2',
             'region','background_father','background_mother','itch','grew','hurt','changed','bleed','elevation']
NUM_COLS = ['age','fitspatrick','diameter_1','diameter_2']
CAT_COLS = [c for c in META_COLS if c not in NUM_COLS]

def preprocess_metadata(dtr, dva, dte, mcols, ncols, ccols):
    tr, va, te = dtr[mcols].copy(), dva[mcols].copy(), dte[mcols].copy()
    sc = MinMaxScaler()
    tr[ncols] = sc.fit_transform(tr[ncols]); va[ncols] = sc.transform(va[ncols]); te[ncols] = sc.transform(te[ncols])
    for df in [tr, va, te]:
        for col in df.select_dtypes(include='bool').columns:
            df[col] = df[col].astype(int)
    scats = [c for c in ccols if tr[c].dtype == object]
    tr = pd.get_dummies(tr, columns=scats)
    va = pd.get_dummies(va, columns=scats).reindex(columns=tr.columns, fill_value=0)
    te = pd.get_dummies(te, columns=scats).reindex(columns=tr.columns, fill_value=0)
    return tr.values.astype(np.float32), va.values.astype(np.float32), te.values.astype(np.float32)

M_train, M_val, M_test = preprocess_metadata(df_train, df_val, df_test, META_COLS, NUM_COLS, CAT_COLS)
meta_dim = M_train.shape[1]
print(f'Metadata dim: {meta_dim}')

def make_loader(c, d, m, y, bs=64, shuffle=False):
    ds = TensorDataset(torch.tensor(c, dtype=torch.float32),
                       torch.tensor(d, dtype=torch.float32),
                       torch.tensor(m, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

train_loader = make_loader(clip_train, dino_train, M_train, y_train, shuffle=True)
val_loader   = make_loader(clip_val,   dino_val,   M_val,   y_val)
test_loader  = make_loader(clip_test,  dino_test,  M_test,  y_test)
print('Loaders listos')

Metadata dim: 69
Loaders listos


## 2. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss binaria — baja el peso de ejemplos fáciles, prioriza difíciles."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p   = torch.sigmoid(logits)
        pt  = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - pt) ** self.gamma * bce
        return loss.mean()

print('Focal Loss definida')

Focal Loss definida


## 3. Arquitecturas: concatenación y atención

In [ ]:
class FusionConcat(nn.Module):
    """Fusión intermedia por concatenación (arquitectura base)."""
    def __init__(self, clip_dim=768, dino_dim=1024, meta_dim=None, proj_dim=256, dropout=0.3):
        super().__init__()
        self.dino_proj = nn.Sequential(nn.Linear(dino_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.clip_proj = nn.Sequential(nn.Linear(clip_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.meta_mlp  = nn.Sequential(nn.Linear(meta_dim, 128), nn.ReLU(), nn.Dropout(dropout),
                                       nn.Linear(128, proj_dim), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(proj_dim*3, 256), nn.ReLU(), nn.Dropout(dropout),
                                        nn.Linear(256, 1))
    def forward(self, x_clip, x_dino, x_meta):
        f = torch.cat([self.dino_proj(x_dino), self.clip_proj(x_clip), self.meta_mlp(x_meta)], dim=1)
        return self.classifier(f).squeeze(1)

class FusionAttention(nn.Module):
    """Fusión con atención — aprende pesos para cada modalidad."""
    def __init__(self, clip_dim=768, dino_dim=1024, meta_dim=None, proj_dim=256, dropout=0.3):
        super().__init__()
        self.dino_proj = nn.Sequential(nn.Linear(dino_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.clip_proj = nn.Sequential(nn.Linear(clip_dim, proj_dim), nn.ReLU(), nn.Dropout(dropout))
        self.meta_mlp  = nn.Sequential(nn.Linear(meta_dim, 128), nn.ReLU(), nn.Dropout(dropout),
                                       nn.Linear(128, proj_dim), nn.ReLU())
        # Red de atención: calcula un peso por modalidad
        self.attn = nn.Sequential(nn.Linear(proj_dim, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Linear(proj_dim, 256), nn.ReLU(), nn.Dropout(dropout),
                                        nn.Linear(256, 1))
    def forward(self, x_clip, x_dino, x_meta):
        f_dino = self.dino_proj(x_dino)
        f_clip = self.clip_proj(x_clip)
        f_clin = self.meta_mlp(x_meta)
        # Stack: [B, 3, proj_dim]
        stack = torch.stack([f_dino, f_clip, f_clin], dim=1)
        # Pesos de atención por modalidad
        w = torch.softmax(self.attn(stack).squeeze(-1), dim=1)  # [B, 3]
        # Suma ponderada
        fused = (stack * w.unsqueeze(-1)).sum(dim=1)  # [B, proj_dim]
        return self.classifier(fused).squeeze(1)

print('Arquitecturas definidas')

Arquitecturas definidas


## 4. Función de entrenamiento genérica

In [ ]:
def train_and_eval(model, criterion, lr, nombre, epochs=50, patience=12, eval_test=False):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    best_auc, pat = 0, 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for xc, xd, xm, yb in train_loader:
            xc, xd, xm, yb = xc.to(device), xd.to(device), xm.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xc, xd, xm), yb)
            loss.backward(); optimizer.step()

        model.eval()
        vloss, vprobs, vlabels = 0, [], []
        with torch.no_grad():
            for xc, xd, xm, yb in val_loader:
                xc, xd, xm, yb = xc.to(device), xd.to(device), xm.to(device), yb.to(device)
                out = model(xc, xd, xm)
                vloss += criterion(out, yb).item()
                vprobs.extend(torch.sigmoid(out).cpu().numpy())
                vlabels.extend(yb.cpu().numpy())
        vloss /= len(val_loader)
        vauc = roc_auc_score(vlabels, vprobs)
        scheduler.step(vloss)

        if vauc > best_auc:
            best_auc = vauc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                break

    print(f'  {nombre:45s} → Val AUC: {best_auc:.4f}')

    if eval_test:
        model.load_state_dict(best_state)
        model.eval()
        tprobs, tlabels = [], []
        with torch.no_grad():
            for xc, xd, xm, yb in test_loader:
                out = model(xc.to(device), xd.to(device), xm.to(device))
                tprobs.extend(torch.sigmoid(out).cpu().numpy())
                tlabels.extend(yb.numpy())
        tprobs, tlabels = np.array(tprobs), np.array(tlabels)
        tpreds = (tprobs >= 0.5).astype(int)
        return best_auc, best_state, {
            'auc': roc_auc_score(tlabels, tprobs),
            'accuracy': accuracy_score(tlabels, tpreds),
            'precision': precision_score(tlabels, tpreds),
            'recall': recall_score(tlabels, tpreds),
            'f1': f1_score(tlabels, tpreds),
            'preds': tpreds, 'labels': tlabels
        }
    return best_auc, best_state, None


## 5. Mejora 1 — Focal Loss (concatenación + focal)
Comparamos contra la base (BCE) usando solo validación.

In [ ]:
print('=== MEJORA 1: Focal Loss ===')
configs_focal = []

# Base con BCE (referencia)
m = FusionConcat(meta_dim=meta_dim, proj_dim=256, dropout=0.3)
auc_bce, _, _ = train_and_eval(m, nn.BCEWithLogitsLoss(), 1e-3, 'Base (BCE)')
configs_focal.append(('Base BCE', auc_bce))

# Focal con distintos gamma
for gamma in [1.0, 2.0, 3.0]:
    m = FusionConcat(meta_dim=meta_dim, proj_dim=256, dropout=0.3)
    auc, _, _ = train_and_eval(m, FocalLoss(alpha=0.25, gamma=gamma), 1e-3, f'Focal (gamma={gamma})')
    configs_focal.append((f'Focal g={gamma}', auc))

print('\nResumen Focal (por Val AUC):')
for nombre, auc in sorted(configs_focal, key=lambda x: -x[1]):
    print(f'  {nombre:20s}: {auc:.4f}')

=== MEJORA 1: Focal Loss ===
  Base (BCE)                                    → Val AUC: 0.9708
  Focal (gamma=1.0)                             → Val AUC: 0.9745
  Focal (gamma=2.0)                             → Val AUC: 0.9750
  Focal (gamma=3.0)                             → Val AUC: 0.9738

Resumen Focal (por Val AUC):
  Focal g=2.0         : 0.9750
  Focal g=1.0         : 0.9745
  Focal g=3.0         : 0.9738
  Base BCE            : 0.9708


## 6. Mejora 2 — Tuning de hiperparámetros

In [ ]:
print('=== MEJORA 2: Tuning de hiperparámetros ===')
configs_tuning = []

for proj_dim in [128, 256, 512]:
    for dropout in [0.2, 0.4]:
        for lr in [1e-3, 5e-4]:
            m = FusionConcat(meta_dim=meta_dim, proj_dim=proj_dim, dropout=dropout)
            nombre = f'proj={proj_dim}, drop={dropout}, lr={lr}'
            auc, state, _ = train_and_eval(m, nn.BCEWithLogitsLoss(), lr, nombre)
            configs_tuning.append((nombre, auc, proj_dim, dropout, lr))

configs_tuning.sort(key=lambda x: -x[1])
print('\nTop 3 configuraciones (por Val AUC):')
for nombre, auc, _, _, _ in configs_tuning[:3]:
    print(f'  {nombre:40s}: {auc:.4f}')

best_tuning = configs_tuning[0]
print(f'\nMejor: {best_tuning[0]}')

=== MEJORA 2: Tuning de hiperparámetros ===
  proj=128, drop=0.2, lr=0.001                  → Val AUC: 0.9698
  proj=128, drop=0.2, lr=0.0005                 → Val AUC: 0.9723
  proj=128, drop=0.4, lr=0.001                  → Val AUC: 0.9702
  proj=128, drop=0.4, lr=0.0005                 → Val AUC: 0.9693
  proj=256, drop=0.2, lr=0.001                  → Val AUC: 0.9731
  proj=256, drop=0.2, lr=0.0005                 → Val AUC: 0.9712
  proj=256, drop=0.4, lr=0.001                  → Val AUC: 0.9730
  proj=256, drop=0.4, lr=0.0005                 → Val AUC: 0.9698
  proj=512, drop=0.2, lr=0.001                  → Val AUC: 0.9730
  proj=512, drop=0.2, lr=0.0005                 → Val AUC: 0.9741
  proj=512, drop=0.4, lr=0.001                  → Val AUC: 0.9742
  proj=512, drop=0.4, lr=0.0005                 → Val AUC: 0.9743

Top 3 configuraciones (por Val AUC):
  proj=512, drop=0.4, lr=0.0005           : 0.9743
  proj=512, drop=0.4, lr=0.001            : 0.9742
  proj=512, drop=0.2, lr

## 7. Mejora 3 — Atención entre modalidades

In [ ]:
print('=== MEJORA 3: Atención entre modalidades ===')
configs_attn = []

for proj_dim in [128, 256]:
    for dropout in [0.2, 0.3]:
        m = FusionAttention(meta_dim=meta_dim, proj_dim=proj_dim, dropout=dropout)
        nombre = f'attn proj={proj_dim}, drop={dropout}'
        auc, state, _ = train_and_eval(m, nn.BCEWithLogitsLoss(), 1e-3, nombre)
        configs_attn.append((nombre, auc, proj_dim, dropout))

configs_attn.sort(key=lambda x: -x[1])
print('\nTop configuraciones atención (por Val AUC):')
for nombre, auc, _, _ in configs_attn[:3]:
    print(f'  {nombre:40s}: {auc:.4f}')

=== MEJORA 3: Atención entre modalidades ===
  attn proj=128, drop=0.2                       → Val AUC: 0.9724
  attn proj=128, drop=0.3                       → Val AUC: 0.9740
  attn proj=256, drop=0.2                       → Val AUC: 0.9712
  attn proj=256, drop=0.3                       → Val AUC: 0.9734

Top configuraciones atención (por Val AUC):
  attn proj=128, drop=0.3                 : 0.9740
  attn proj=256, drop=0.3                 : 0.9734
  attn proj=128, drop=0.2                 : 0.9724


## 8. Selección del mejor modelo y evaluación FINAL en test
Solo aquí tocamos el test — con el ganador por validación.

In [ ]:
# Recolectar el mejor de cada estrategia (por Val AUC)
candidatos = {
    'Focal':   max(configs_focal,  key=lambda x: x[1])[1],
    'Tuning':  configs_tuning[0][1],
    'Atención': configs_attn[0][1],
}
print('Mejor Val AUC por estrategia:')
for k, v in candidatos.items():
    print(f'  {k:10s}: {v:.4f}')

ganador = max(candidatos, key=candidatos.get)
print(f'\nEstrategia ganadora (por validación): {ganador}')

Mejor Val AUC por estrategia:
  Focal     : 0.9750
  Tuning    : 0.9743
  Atención  : 0.9740

Estrategia ganadora (por validación): Focal


In [ ]:
# Re-entrenar el ganador y evaluar en TEST (una sola vez)
print(f'\nEntrenando modelo ganador ({ganador}) y evaluando en test...\n')

if ganador == 'Tuning':
    _, _, pd_, dr_, lr_ = configs_tuning[0]
    model = FusionConcat(meta_dim=meta_dim, proj_dim=pd_, dropout=dr_)
    crit, lr_final = nn.BCEWithLogitsLoss(), lr_
elif ganador == 'Atención':
    _, _, pd_, dr_ = configs_attn[0]
    model = FusionAttention(meta_dim=meta_dim, proj_dim=pd_, dropout=dr_)
    crit, lr_final = nn.BCEWithLogitsLoss(), 1e-3
else:  # Focal
    best_focal = max(configs_focal, key=lambda x: x[1])[0]
    gamma = float(best_focal.split('=')[-1]) if 'g=' in best_focal else 2.0
    model = FusionConcat(meta_dim=meta_dim, proj_dim=256, dropout=0.3)
    crit, lr_final = FocalLoss(alpha=0.25, gamma=gamma), 1e-3

_, best_state, test_metrics = train_and_eval(model, crit, lr_final, f'GANADOR ({ganador})', eval_test=True)

print('\n' + '='*55)
print(f'  FUSIÓN PROPUESTA MEJORADA ({ganador})')
print('='*55)
print(f"  AUC-ROC   : {test_metrics['auc']:.4f}")
print(f"  Accuracy  : {test_metrics['accuracy']*100:.1f}%")
print(f"  Precision : {test_metrics['precision']*100:.1f}%")
print(f"  Recall    : {test_metrics['recall']*100:.1f}%")
print(f"  F1-Score  : {test_metrics['f1']:.4f}")
print('='*55)
print()
print(classification_report(test_metrics['labels'], test_metrics['preds'],
                            target_names=['Benigno', 'Maligno']))


Entrenando modelo ganador (Focal) y evaluando en test...

  GANADOR (Focal)                               → Val AUC: 0.9733

  FUSIÓN PROPUESTA MEJORADA (Focal)
  AUC-ROC   : 0.9733
  Accuracy  : 90.4%
  Precision : 94.0%
  Recall    : 85.5%
  F1-Score  : 0.8952

              precision    recall  f1-score   support

     Benigno       0.88      0.95      0.91       120
     Maligno       0.94      0.85      0.90       110

    accuracy                           0.90       230
   macro avg       0.91      0.90      0.90       230
weighted avg       0.91      0.90      0.90       230



## 9. Comparación: propuesta base vs mejorada

In [ ]:
comparacion = pd.DataFrame([
    {'version': 'Propuesta base (concat + BCE)', 'auc': 0.9571, 'accuracy': 0.8913,
     'precision': 0.8761, 'recall': 0.9000, 'f1': 0.8879},
    {'version': f'Propuesta mejorada ({ganador})', 'auc': round(test_metrics['auc'],4),
     'accuracy': round(test_metrics['accuracy'],4), 'precision': round(test_metrics['precision'],4),
     'recall': round(test_metrics['recall'],4), 'f1': round(test_metrics['f1'],4)},
])
print('=== COMPARACIÓN BASE vs MEJORADA ===')
print(comparacion.to_string(index=False))

comparacion.to_csv(os.path.join(RESULTS_PATH, 'comparacion_propuesta_mejorada.csv'), index=False)
print('\n Guardado')

delta_auc = test_metrics['auc'] - 0.9571
if delta_auc > 0:
    print(f'\n La mejora subió el AUC en {delta_auc:.4f} ({delta_auc*100:.2f} puntos)')
else:
    print(f'\n La versión base sigue siendo mejor o igual ({delta_auc:.4f}).')
    print('   Esto también es un resultado válido — documenta ambas en la tesis.')

=== COMPARACIÓN BASE vs MEJORADA ===
                      version    auc  accuracy  precision  recall     f1
Propuesta base (concat + BCE) 0.9571    0.8913     0.8761  0.9000 0.8879
   Propuesta mejorada (Focal) 0.9733    0.9043     0.9400  0.8545 0.8952

 Guardado

 La mejora subió el AUC en 0.0162 (1.62 puntos)
